<a href="https://colab.research.google.com/github/sayleepradhan/ai-projects/blob/main/rag_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Pipeline from Scratch

| Component | Choice |
| --- | --- |
| **Dataset** | Wikipedia AI/ML articles via the MediaWiki API |
| **Embeddings** | `sentence-transformers/all-MiniLM-L6-v2` (local, free) |
| **Vector Store** | FAISS (local, no account needed) |
| **Generation** | Claude via Anthropic API |

This notebook builds a retrieval-augmented generation (RAG) pipeline from scratch — no LangChain, no LlamaIndex. The goal is to understand each component before using frameworks that abstract them away.

**Pipeline Overview**

1. Load and filter Wikipedia articles on AI/ML topics
2. Chunk text into manageable segments
3. Generate embeddings using a local sentence-transformer model
4. Index embeddings in FAISS for fast similarity search
5. Retrieve relevant chunks for a user query
6. Augment a prompt with retrieved context and generate an answer via Claude

## 1. Install Dependencies

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu anthropic pandas tqdm

## 2. Configuration

In [ ]:
import os

# Set your Anthropic API key
os.environ["ANTHROPIC_API_KEY"] = "<API_KEY>"

# Topics to filter from Wikipedia
# These are exact or partial matches against article titles
AI_TOPICS = [
    "large language model",
    "transformer",
    "retrieval-augmented generation",
    "generative adversarial network",
    "diffusion model",
    "reinforcement learning",
    "natural language processing",
    "neural network",
    "attention mechanism",
    "word embedding",
    "BERT",
    "GPT",
    "fine-tuning",
    "vector database",
    "prompt engineering",
    "reinforcement learning from human feedback"
]

CHUNK_SIZE = 1024       # characters per chunk
TOP_K = 3               # number of chunks to retrieve
MAX_ARTICLES = 50       # cap to keep runtime reasonable on CPU

print("Configuration set.")

Configuration set.


## 3. Load Wikipedia Articles

We fetch AI/ML-related articles directly from the MediaWiki API. Each article is retrieved by its exact title, giving us clean plaintext extracts without needing to download the full Wikipedia dump (~20 GB).

In [ ]:
import requests
import time
import pandas as pd

AI_ARTICLE_TITLES = [
    "Large language model",
    "Transformer (machine learning model)",
    "Retrieval-augmented generation",
    "Generative adversarial network",
    "Diffusion model",
    "Reinforcement learning",
    "Natural language processing",
    "Artificial neural network",
    "Attention (machine learning)",
    "Word embedding",
    "BERT (language model)",
    "Generative pre-trained transformer",
    "Fine-tuning (deep learning)",
    "Vector database",
    "Prompt engineering",
    "Reinforcement learning from human feedback"
]

HEADERS = {
    "User-Agent": "RAGPortfolioProject/1.0 (sayleepradhan.dev; for educational use)"
}

def fetch_wikipedia_article(title: str, retries: int = 3) -> dict | None:
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
        "redirects": True
    }

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=10)
            # Check we actually got a response body before parsing
            if not response.text.strip():
                print(f"  Empty response for '{title}', attempt {attempt+1}/{retries}")
                time.sleep(2)
                continue

            data = response.json()
            pages = data["query"]["pages"]
            page = next(iter(pages.values()))

            if "missing" in page:
                print(f"  Not found: {title}")
                return None

            return {"title": page["title"], "text": page["extract"]}

        except Exception as e:
            print(f"  Error fetching '{title}' (attempt {attempt+1}/{retries}): {e}")
            time.sleep(2)

    print(f"  Failed after {retries} attempts: {title}")
    return None

articles = []
print("Fetching Wikipedia articles...")
for title in AI_ARTICLE_TITLES:
    article = fetch_wikipedia_article(title)
    if article:
        articles.append(article)
        print(f"  Fetched: {article['title']} ({len(article['text'])} chars)")
    time.sleep(0.5)  # be polite to the API

print(f"\nTotal articles fetched: {len(articles)}")

Fetching Wikipedia articles...
  Fetched: Large language model (65112 chars)
  Fetched: Transformer (deep learning) (112661 chars)
  Fetched: Retrieval-augmented generation (10498 chars)
  Fetched: Generative adversarial network (115716 chars)
  Fetched: Diffusion model (181476 chars)
  Fetched: Reinforcement learning (52180 chars)
  Fetched: Natural language processing (32402 chars)
  Fetched: Neural network (machine learning) (63096 chars)
  Fetched: Attention (machine learning) (25514 chars)
  Fetched: Word embedding (10018 chars)
  Fetched: BERT (language model) (17538 chars)
  Fetched: Generative pre-trained transformer (13888 chars)
  Fetched: Fine-tuning (deep learning) (5369 chars)
  Fetched: Vector database (3595 chars)
  Fetched: Prompt engineering (18828 chars)
  Fetched: Reinforcement learning from human feedback (100322 chars)

Total articles fetched: 16


## 4. Chunking

We split each article into chunks of approximately 1,024 characters. Rather than cutting at a fixed character count (which can split mid-sentence and hurt retrieval quality), we split on sentence boundaries. The function accumulates sentences until the next one would exceed the limit, then starts a new chunk.

In [ ]:
import re
import pandas as pd

def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    """
    Split text into chunks of approximately chunk_size characters.
    Splits on sentence boundaries to avoid cutting mid-sentence.
    """
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current = [], ""

    for sentence in sentences:
        if len(current) + len(sentence) <= chunk_size:
            current += " " + sentence
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sentence
    if current.strip():
        chunks.append(current.strip())
    return chunks

# Build the chunk dataframe
rows = []
for article in articles:
    for chunk in split_into_chunks(article["text"]):
        rows.append({"title": article["title"], "chunk": chunk})

df = pd.DataFrame(rows)
print(f"Total chunks: {len(df)}")
print(f"Average chunk length: {df['chunk'].str.len().mean():.0f} characters")
df.head(3)

Total chunks: 685
Average chunk length: 1208 characters


,title,chunk
0,Large language model,A large language model (LLM) is a computationa...
1,Large language model,LLMs represent a significant new technology in...
2,Large language model,Reinforcement learning from human feedback (RL...


## 5. Generate Embeddings

We use `all-MiniLM-L6-v2` from the `sentence-transformers` library. This is a lightweight, open-source model that runs locally with no API calls or costs. It produces 384-dimensional embedding vectors that capture the semantic meaning of each chunk, enabling similarity-based retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

print("Loading embedding model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Generating embeddings (this may take a few minutes on CPU)...")
embeddings = embedder.encode(
    df["chunk"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

df["embedding"] = list(embeddings)
print(f"Embedding shape per chunk: {embeddings[0].shape}")
print("Done.")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings (this may take a few minutes on CPU)...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape per chunk: (384,)
Done.


## 6. Build FAISS Index

Rather than looping over all embeddings and computing cosine similarity one by one, we use FAISS (Facebook AI Similarity Search) for efficient vector lookup. We normalize all vectors to unit length so that inner product is equivalent to cosine similarity, then build a flat inner-product index. `IndexFlatIP` does exact brute-force search, which is fast enough for our 625 vectors and keeps things simple.

In [ ]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]
embeddings_matrix = embeddings.astype("float32")

# Normalize for cosine similarity (inner product on unit vectors = cosine similarity)
faiss.normalize_L2(embeddings_matrix)

# Build a flat inner-product index
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_matrix)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")

FAISS index built with 685 vectors of dimension 384.


## Sanity Check: Cosine Similarity Demo

Before running the full pipeline, let's verify the similarity metric works as expected. We embed a question alongside a relevant source and an irrelevant source, then compare the cosine similarity scores. A well-functioning embedding model should assign a much higher score to the semantically related pair.

In [ ]:
def cosine_similarity_check(text_a: str, text_b: str) -> float:
    """Compute cosine similarity between two texts."""
    emb_a = embedder.encode([text_a], convert_to_numpy=True).astype("float32")
    emb_b = embedder.encode([text_b], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(emb_a)
    faiss.normalize_L2(emb_b)
    return float(np.dot(emb_a[0], emb_b[0]))

question = "What is a transformer model in machine learning?"
good_source = "The transformer is a deep learning architecture based on self-attention mechanisms, widely used in NLP."
bad_source = "The Eiffel Tower is located in Paris, France."

print(f"Question vs relevant source: {cosine_similarity_check(question, good_source):.4f}")
print(f"Question vs irrelevant source: {cosine_similarity_check(question, bad_source):.4f}")

Question vs relevant source: 0.5766
Question vs irrelevant source: 0.0549


## 7. Retrieval Function

Given a user query, we embed it, search the FAISS index, and return the top-K most similar chunks along with their source article titles and similarity scores.

In [ ]:
def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    """
    Retrieve the top-k most relevant chunks for a given query.
    Returns a list of dicts with keys: title, chunk, score.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": df.iloc[idx]["title"],
            "chunk": df.iloc[idx]["chunk"],
            "score": round(float(score), 4)
        })
    return results

# Test retrieval
test_query = "How does the attention mechanism work in transformers?"
results = retrieve(test_query)

for i, r in enumerate(results):
    print(f"\n--- Chunk {i+1} | Source: {r['title']} | Score: {r['score']} ---")
    print(r["chunk"][:300], "...")


--- Chunk 1 | Source: Transformer (deep learning) | Score: 0.6831 ---
This allows the model to capture more complex and long-range dependencies in deeper layers. Many transformer attention heads encode relevance relations that are meaningful to humans. For example, some attention heads can attend mostly to the next word, while others mainly attend from verbs to their  ...

--- Chunk 2 | Source: Attention (machine learning) | Score: 0.6417 ---
In vision, visual attention helps models focus on relevant image regions, enhancing object detection and image captioning.


=== Attention maps as explanations for vision transformers ===

From the original paper on vision transformers (ViT), visualizing attention scores as a heat map (called salien ...

--- Chunk 3 | Source: Transformer (deep learning) | Score: 0.5842 ---
This allows the transformer to take any encoded position and find a linear sum of the encoded locations of its neighbors. This sum of encoded positions, when fed into the atten

## 8. Augmented Generation with Claude

We pass the retrieved chunks as context into Claude's prompt. The system prompt instructs the model to answer only from the provided context, which reduces hallucinations. This is the core benefit of RAG: grounding the LLM's responses in specific, retrieved evidence.

In [ ]:
import anthropic

client = anthropic.Anthropic()

SYSTEM_PROMPT = """You are a knowledgeable assistant that answers questions strictly based on the provided context.
If the answer cannot be found in the context, say: "I don't have enough information in the provided context to answer this."
Do not use any external knowledge beyond what is given."""

USER_PROMPT_TEMPLATE = """Use the context below to answer the question. The context is sourced from Wikipedia articles on AI/ML topics.

<START_OF_CONTEXT>
{context}
<END_OF_CONTEXT>

Question: {question}
Answer:"""

def answer_question(question: str, k: int = TOP_K) -> dict:
    """
    Full RAG pipeline: retrieve relevant chunks, augment prompt, generate answer.
    Returns the answer and the retrieved source chunks for transparency.
    """
    # Step 1: Retrieve
    retrieved = retrieve(question, k=k)

    # Step 2: Build context string
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[Source: {r['title']}]\n{r['chunk']}")
    context = "\n\n".join(context_parts)

    # Step 3: Build prompt
    prompt = USER_PROMPT_TEMPLATE.format(context=context, question=question)

    # Step 4: Generate answer via Claude
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "answer": message.content[0].text,
        "sources": [{"title": r["title"], "score": r["score"]} for r in retrieved]
    }

print("answer_question() function ready.")

answer_question() function ready.


## 9. Test the Full Pipeline

In [ ]:
# Test 1: On-topic question
result = answer_question("What is retrieval-augmented generation and why is it useful?")

print("QUESTION:", result["question"])
print("\nANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s['title']} (score: {s['score']})")

QUESTION: What is retrieval-augmented generation and why is it useful?

ANSWER:
Based on the provided context, retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources before generating responses.

**How RAG works:**
- LLMs first refer to a specified set of documents or external sources
- The system encodes queries and documents into vectors and stores them in a vector database
- When given a query, it retrieves the most relevant documents by finding those with vectors most similar to the query vector
- The LLM then generates responses based on both the original query and the context from the retrieved documents

**Why RAG is useful:**
1. **Access to updated information**: It allows LLMs to use domain-specific and/or updated information that is not available in their pre-existing training data

2. **Overcomes static training limitations**: Unlike LLMs that rely solely on static t

In [ ]:
# Test 2: Another on-topic question
result2 = answer_question("How does reinforcement learning from human feedback (RLHF) work?")

print("QUESTION:", result2["question"])
print("\nANSWER:")
print(result2["answer"])
print("\nSOURCES:")
for s in result2["sources"]:
    print(f"  - {s['title']} (score: {s['score']})")

QUESTION: How does reinforcement learning from human feedback (RLHF) work?

ANSWER:
Based on the provided context, reinforcement learning from human feedback (RLHF) works through the following process:

**Core Approach:**
RLHF is a technique designed to align intelligent agents with human preferences by training a reward model to represent those preferences, which is then used to train other models through reinforcement learning.

**How it differs from classical RL:**
In classical reinforcement learning, agents learn a policy function by optimizing for a predefined reward signal. However, explicitly defining a reward function that accurately captures human preferences is challenging.

**The RLHF Process:**

1. **Reward Model Training**: RLHF first trains a "reward model" directly from human feedback. This model is trained in a supervised manner to predict whether a response to a given prompt is good (high reward) or bad (low reward) based on ranking data collected from human annotators

In [ ]:
# Test 3: Off-topic question -- the model should decline gracefully
result3 = answer_question("What is the capital of France?")

print("QUESTION:", result3["question"])
print("\nANSWER:")
print(result3["answer"])

QUESTION: What is the capital of France?

ANSWER:
I don't have enough information in the provided context to answer this. The context provided discusses natural language processing topics like named entity recognition, sentiment analysis, BERT language models, and transformer architectures, but does not contain information about the capital of France.


## 10. Limitations and What's Next

This pipeline is intentionally simple. A few honest limitations to note:

**Fixed chunk size:** We split on sentence boundaries at ~1,024 characters. More advanced strategies (token-based splitting, overlapping windows, semantic chunking) can meaningfully improve retrieval quality.

**Flat FAISS index:** `IndexFlatIP` does exact search, which is fine up to ~100K vectors. For larger corpora, approximate nearest-neighbor indexes (HNSW, IVF) trade a small amount of accuracy for significantly faster lookups.

**No re-ranking:** We retrieve by embedding similarity only. Adding a cross-encoder re-ranker as a second stage typically improves answer quality by scoring query-chunk pairs more precisely.

**No evaluation:** We eyeballed the results here. A production system would use RAG evaluation metrics like faithfulness, answer relevance, and context recall (e.g., via RAGAS) to measure quality systematically.